# A simple chatbot for a shoe and sandal store.

In [ ]:
#import library
import os
from openai import OpenAI
import gradio as gr
import json

In [ ]:
#setting ollama
BASE_URL = "http://localhost:11434/v1"
MODEL = 'llama3.2:latest'
ollama = OpenAI(base_url=BASE_URL, api_key="api_key")

In [ ]:
#load json
with open('product.json', 'r', encoding='utf-8') as f:
    products = json.load(f)
    
products

In [ ]:
#make a product context
product_context = json.dumps(
    products,
    indent = 4,
    ensure_ascii=False
)

print(product_context)

In [ ]:
system_message = f"""
You are a helpful AI shopping assistant for a shoe store.

Your responsibilities:
- Help customers find suitable shoes.
- Recommend products based on their needs and budget.
- Answer questions about products.
- Only use information from the product catalog.
- Never invent products, prices, sizes, or stock.
- If the information is unavailable, say that you don't know.

Product catalog:

{product_context}
"""

In [ ]:
print(system_message)

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = ollama.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()